In [0]:
import sys
from pathlib import Path

REPO_PATH = Path(
    "/Workspace/Users/tiiina.aminian@gmail.com/Ontario_Electricity_Lakehouse"
)

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

from src.transformation.silver_demand import transform_silver_demand
from src.transformation.silver_validation import validate_silver_demand
from src.transformation.silver_writer import write_silver

BRONZE_TABLE = "ontario_energy.bronze.ieso_demand_2025"
SILVER_TABLE = "ontario_energy.silver.ieso_demand_2025"

# Extract
bronze_df = spark.table(BRONZE_TABLE)

# Transform
silver_df = transform_silver_demand(bronze_df)

# Validate
validate_silver_demand(silver_df)

bronze_count = bronze_df.count()
silver_count = silver_df.count()

if bronze_count != silver_count:
    raise ValueError(
        f"Bronze/Silver row count mismatch: "
        f"{bronze_count} vs {silver_count}"
    )

# Write
write_silver(silver_df, target_table=SILVER_TABLE)

# Post-write verification
persisted_df = spark.table(SILVER_TABLE)

if persisted_df.count() != silver_count:
    raise ValueError("Silver persisted row count mismatch")

validate_silver_demand(persisted_df)

print("Silver processing completed successfully")

Silver processing completed successfully
